# 01 — Data Preprocessing
**QM640 Data Analytics Capstone** | Nandini Nagaraj | Walsh College | Spring 2026

---

## Purpose
This notebook handles all data ingestion and cleaning steps.
It outputs a clean dataset used by  and .

## Steps
1. Install libraries
2. Load raw dataset from GitHub
3. Rename and standardise columns
4. Report and impute missing values
5. Engineer features (log transform, composite index)
6. Check class balance
7. Export clean dataset

> **PLACEHOLDER:** Update  if your CSV path differs.

---
## Section 1: Install Libraries
Run once per Colab session. Takes about 60 seconds.

In [ ]:
!pip install -q pandas numpy matplotlib seaborn geopandas scikit-learn scipy mapclassify requests
print("✅ All libraries installed.")

---
## Section 2: Load Dataset

The cell below loads data from your GitHub repository.

> **PLACEHOLDER:** If your CSV filename or GitHub path differs, update `GITHUB_RAW_URL` below.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── PLACEHOLDER: Update this URL if your CSV is in a subfolder (e.g. /data/dpi_state_data.csv) ──
GITHUB_RAW_URL = "https://raw.githubusercontent.com/NandiniJu/dpi-capstone-qm640/main/dpi_state_data.csv"

df = pd.read_csv(GITHUB_RAW_URL)

print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print(f"Column names: {df.columns.tolist()}")
df.head()

---
## Section 3: Data Cleaning

Steps performed:
- Rename columns for consistency
- Check and report missingness
- Impute missing values (median by state)
- Log-transform `per_capita_income` to reduce right skew
- Reconstruct `digital_adoption_index` from normalized components
- Report class balance

> **PLACEHOLDER:** If your column names differ from what was defined in the data dictionary, update the `COLUMN_MAP` dictionary below.

In [ ]:
# ── PLACEHOLDER: Map your actual column names to the standard names used throughout this notebook ──
# If your CSV already uses these exact names, leave this dictionary empty: {}
COLUMN_MAP = {
    'per_capita_income_inr': 'per_capita_income',   # rename if needed
    # 'your_col_name': 'standard_name',             # add more as needed
}

df.rename(columns=COLUMN_MAP, inplace=True)
print("Columns after rename:", df.columns.tolist())

In [ ]:
# Missingness report
missing_df = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %':     (df.isnull().sum() / len(df) * 100).round(2)
})
print("=== Missingness Report ===")
print(missing_df[missing_df['Missing Count'] > 0].to_string() or "No missing values found.")

In [ ]:
# Impute missing values using within-state median
NUMERIC_COLS = [
    'upi_txn_per_capita', 'internet_penetration', 'aadhaar_coverage',
    'literacy_rate', 'per_capita_income', 'urbanization_rate'
]

for col in NUMERIC_COLS:
    if df[col].isnull().sum() > 0:
        before = df[col].isnull().sum()
        df[col] = df.groupby('state_name')[col].transform(lambda x: x.fillna(x.median()))
        df[col].fillna(df[col].median(), inplace=True)  # fallback: global median
        print(f"  {col}: imputed {before} missing values")

print("\n✅ Imputation complete. Remaining missing values:", df[NUMERIC_COLS].isnull().sum().sum())

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Log-transform per_capita_income
df['log_per_capita_income'] = np.log(df['per_capita_income'])

# Reconstruct digital_adoption_index from scratch
# (unweighted mean of min-max normalized digital indicators)
DIGITAL_COLS = ['upi_txn_per_capita', 'internet_penetration', 'aadhaar_coverage']
scaler = MinMaxScaler()
normalized = scaler.fit_transform(df[DIGITAL_COLS])
df['digital_adoption_index'] = normalized.mean(axis=1)

print("✅ Feature engineering complete.")
print(f"  log_per_capita_income range: {df['log_per_capita_income'].min():.2f} – {df['log_per_capita_income'].max():.2f}")
print(f"  digital_adoption_index range: {df['digital_adoption_index'].min():.3f} – {df['digital_adoption_index'].max():.3f}")

In [ ]:
# Class balance check
counts = df['performance_category'].value_counts()
print("=== Class Balance ===")
print(f"  Low-performing  (0): {counts.get(0, 0)} observations ({counts.get(0,0)/len(df)*100:.1f}%)")
print(f"  High-performing (1): {counts.get(1, 0)} observations ({counts.get(1,0)/len(df)*100:.1f}%)")
print(f"\n  Total observations: {len(df)}")
print(f"  Dataset is {'balanced' if abs(counts.get(0,0) - counts.get(1,0)) / len(df) < 0.1 else 'imbalanced — consider SMOTE or class weights'}")

In [ ]:
# Export cleaned dataset for use in 02_eda.ipynb and 03_modeling.ipynb
df.to_csv("dpi_state_clean.csv", index=False)
print(f"Exported clean dataset: {len(df)} rows x {df.shape[1]} columns")
print("Saved as: dpi_state_clean.csv")
print("Final columns:", df.columns.tolist())
df.describe().round(2)

---
## Download Clean Dataset
Download  to upload to your GitHub repo under .

In [ ]:
from google.colab import files
files.download("dpi_state_clean.csv")
print("⬇️  Download triggered — check your browser downloads folder.")